In [7]:
import pandas as pd

# Load the FAO SDG 2.1.1 dataset
df = pd.read_csv("FAO,DF_SDG_2_1_1,1.0+all.csv")

# Filter for target regions
target_regions = ["Central Asia", "Central America", "South America"]
df = df[df["Area"].isin(target_regions)]

# Keep only percentage-based data
df = df[df["UNIT_MEASURE"] == "PT"]

# Select and rename columns
df = df[["Area", "TIME_PERIOD", "OBS_VALUE"]]
df.columns = ["Region", "Year", "Undernourishment (%)"]

# Convert Year to integer
df["Year"] = df["Year"].astype(int)

# Clean Undernourishment (%) column
df['Undernourishment (%)'] = df['Undernourishment (%)'].replace('<', '', regex=True)
df['Undernourishment (%)'] = pd.to_numeric(df['Undernourishment (%)'], errors='coerce')

# Add lagged feature
df['Last_Year_Undernourishment'] = df.groupby('Region')['Undernourishment (%)'].shift(1)
df['Last_Year_Undernourishment'] = df.groupby('Region')['Last_Year_Undernourishment'].fillna(method='bfill')

# Train-Test Split (70-30: 2001-2016 train, 2017-2023 test)
train_end = 2016
test_start = 2017
test_end = 2023

region_data = {}
for region in target_regions:
    region_df = df[df['Region'] == region].copy()
    train_data = region_df[region_df['Year'] <= train_end]
    test_data = region_df[(region_df['Year'] >= test_start) & (region_df['Year'] <= test_end)]
    region_data[region] = {
        'full_data': region_df,
        'train': train_data,
        'test': test_data
    }

# Verify
for region, data in region_data.items():
    print(f"\n{region} Training Data (2001-2016):")
    print(data['train'])
    print(f"\n{region} Test Data (2017-2023):")
    print(data['test'])

# Optional: Print data availability details
for region in target_regions:
    print(f"\n{region} Data Availability:")
    print(f"Total years of data: {region_data[region]['full_data']['Year'].nunique()}")
    print(f"Year range: {region_data[region]['full_data']['Year'].min()} - {region_data[region]['full_data']['Year'].max()}")


Central Asia Training Data (2001-2016):
            Region  Year  Undernourishment (%)  Last_Year_Undernourishment
9651  Central Asia  2000                  14.5                        14.5
9652  Central Asia  2001                  16.6                        14.5
9653  Central Asia  2002                  17.3                        16.6
9654  Central Asia  2003                  16.1                        17.3
9655  Central Asia  2004                  15.7                        16.1
9656  Central Asia  2005                  13.8                        15.7
9657  Central Asia  2006                  12.0                        13.8
9658  Central Asia  2007                   9.9                        12.0
9659  Central Asia  2008                   8.6                         9.9
9660  Central Asia  2009                   7.4                         8.6
9661  Central Asia  2010                   6.4                         7.4
9662  Central Asia  2011                   5.6             

C:\Users\shahe\AppData\Local\Temp\ipykernel_12124\524765819.py:26: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df['Last_Year_Undernourishment'] = df.groupby('Region')['Last_Year_Undernourishment'].fillna(method='bfill')
C:\Users\shahe\AppData\Local\Temp\ipykernel_12124\524765819.py:26: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Last_Year_Undernourishment'] = df.groupby('Region')['Last_Year_Undernourishment'].fillna(method='bfill')


In [9]:
# SARIMA model for Central Asia
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Filter training and test data
train = df[(df["Region"] == "Central Asia") & (df["Year"] <= 2015)].copy()
test = df[(df["Region"] == "Central Asia") & (df["Year"] >= 2016)].copy()
train["Undernourishment (%)"] = train["Undernourishment (%)"].astype(float)

# Check data characteristics
print("Central Asia data characteristics:")
print(f"Training data points: {len(train)}")
print(f"Range: {train['Undernourishment (%)'].min():.2f} to {train['Undernourishment (%)'].max():.2f}")
print(f"Mean: {train['Undernourishment (%)'].mean():.2f}")

# Fit SARIMA with enhanced seasonal configuration
model = auto_arima(
    train["Undernourishment (%)"],
    seasonal=True,       # Enable seasonality
    m=1,                 # Annual data
    start_p=0, max_p=3,  # Control AR terms
    start_q=0, max_q=3,  # Control MA terms
    d=None,              # Auto-detect differencing
    trace=True,          # Show model selection process
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True        # Use stepwise selection for efficiency
)

# Print model details
print(f"\nBest model: {model.order}")

# Forecast for 10 years
forecast = model.predict(n_periods=10)

# Create forecast DataFrame
central_asia_sarima_forecast = pd.DataFrame({
    "Year": list(range(2016, 2026)),
    "SARIMA": forecast
})

Central Asia data characteristics:
Training data points: 16
Range: 3.90 to 17.30
Mean: 10.12
Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=47.337, Time=0.03 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=42.254, Time=0.04 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=44.765, Time=0.07 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=51.161, Time=0.03 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=44.213, Time=0.13 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=44.232, Time=0.11 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=46.209, Time=0.21 sec
 ARIMA(1,1,0)(0,0,0)[0]             : AIC=40.319, Time=0.06 sec
 ARIMA(2,1,0)(0,0,0)[0]             : AIC=42.303, Time=0.07 sec
 ARIMA(1,1,1)(0,0,0)[0]             : AIC=42.310, Time=0.10 sec
 ARIMA(0,1,1)(0,0,0)[0]             : AIC=45.828, Time=0.05 sec
 ARIMA(2,1,1)(0,0,0)[0]             : AIC=44.246, Time=0.16 sec

Best model:  ARIMA(1,1,0)(0,0,0)[0]          
Total fit time: 1.112 seconds

Best model: (1, 1,

C:\Users\shahe\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\shahe\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [10]:
# Evaluate on available actual values (2017-2023)
eval_df = df[(df["Region"] == "Central Asia") & df["Year"].between(2017, 2023)].copy()
eval_df["Undernourishment (%)"] = eval_df["Undernourishment (%)"].astype(float)

# Add SARIMA predictions
eval_df = eval_df.merge(
    central_asia_sarima_forecast[["Year", "SARIMA"]],
    on="Year",
    how="left"
)

# Create naive model (last observed value)
last_value = train["Undernourishment (%)"].iloc[-1]
eval_df["Naive"] = last_value

# Calculate metrics for SARIMA
y_true = eval_df["Undernourishment (%)"]
y_pred_sarima = eval_df["SARIMA"]
y_pred_naive = eval_df["Naive"]

# SARIMA metrics
mae_sarima = mean_absolute_error(y_true, y_pred_sarima)
rmse_sarima = mean_squared_error(y_true, y_pred_sarima, squared=False)

# R-squared calculation with robust handling
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean)**2)
ss_res = np.sum((y_true - y_pred_sarima)**2)

try:
    r2_sarima = 1 - (ss_res / ss_tot)
except ZeroDivisionError:
    r2_sarima = np.nan

# Naive metrics
mae_naive = mean_absolute_error(y_true, y_pred_naive)
rmse_naive = mean_squared_error(y_true, y_pred_naive, squared=False)

# Print metrics
print("Central Asia - SARIMA Evaluation Metrics")
print(f"MAE: {mae_sarima:.4f}")
print(f"RMSE: {rmse_sarima:.4f}")
print(f"R²: {r2_sarima:.4f}")

# Compare with naive model
print("\nNaive model metrics:")
print(f"MAE: {mae_naive:.4f}")
print(f"RMSE: {rmse_naive:.4f}")

# Create comparison table
results = pd.DataFrame({
    "Year": eval_df["Year"],
    "Actual": y_true,
    "SARIMA": y_pred_sarima,
    "Error": y_true - y_pred_sarima
})
print("\nYear-by-year comparison:")
print(results[["Year", "Actual", "SARIMA", "Error"]])

Central Asia - SARIMA Evaluation Metrics
MAE: 0.3370
RMSE: 0.3615
R²: -1.3038

Naive model metrics:
MAE: 0.8429
RMSE: 0.8759

Year-by-year comparison:
   Year  Actual    SARIMA     Error
0  2017     3.4  3.327072  0.072928
1  2018     2.9  3.124535 -0.224535
2  2019     2.6  2.963105 -0.363105
3  2020     3.2  2.834437  0.365563
4  2021     3.2  2.731884  0.468116
5  2022     3.1  2.650145  0.449855
6  2023     3.0  2.584995  0.415005


C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [11]:
# SARIMA model for Central America
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Prepare Central America training data (2001-2015)
train = df[(df["Region"] == "Central America") & (df["Year"] >= 2001) & (df["Year"] <= 2015)].copy()
test = df[(df["Region"] == "Central America") & (df["Year"] >= 2016)].copy()
train["Undernourishment (%)"] = train["Undernourishment (%)"].astype(float)

# Check training data
print("Central America training data summary:")
print(f"Number of years: {len(train)}")
print(f"Years: {min(train['Year'])} to {max(train['Year'])}")
print(f"Undernourishment range: {min(train['Undernourishment (%)']):.2f} to {max(train['Undernourishment (%)']):.2f}")

# Fit SARIMA model with seasonal optimization
model = auto_arima(
    train["Undernourishment (%)"],
    seasonal=True,      # Enable seasonality
    m=1,                # Annual data
    start_p=0, max_p=2, # Limit model complexity
    start_q=0, max_q=2,
    max_order=5,        # Limit total parameters
    trace=True,         # Show models tested
    suppress_warnings=True,
    stepwise=True,      # Faster search
    error_action='ignore'
)

# Print best model details
print(f"\nBest SARIMA model: {model.order}")
print(f"AIC: {model.aic()}")

# Forecast 10 years ahead (2016-2025)
forecast = model.predict(n_periods=10)

# Save forecasts to DataFrame
central_america_sarima_forecast = pd.DataFrame({
    "Year": list(range(2016, 2026)),
    "SARIMA": forecast
})

# Print forecast samples
print("\nForecast samples:")
print(f"2016 forecast: {central_america_sarima_forecast['SARIMA'].iloc[0]:.2f}")
print(f"2020 forecast: {central_america_sarima_forecast['SARIMA'].iloc[4]:.2f}")
print(f"2025 forecast: {central_america_sarima_forecast['SARIMA'].iloc[9]:.2f}")

Central America training data summary:
Number of years: 15
Years: 2001 to 2015
Undernourishment range: 6.20 to 8.20
Performing stepwise search to minimize aic
 ARIMA(0,0,0)(0,0,0)[0] intercept   : AIC=29.712, Time=0.02 sec
 ARIMA(1,0,0)(0,0,0)[0] intercept   : AIC=24.411, Time=0.11 sec
 ARIMA(0,0,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.16 sec
 ARIMA(0,0,0)(0,0,0)[0]             : AIC=102.788, Time=0.02 sec
 ARIMA(2,0,0)(0,0,0)[0] intercept   : AIC=24.475, Time=0.12 sec
 ARIMA(1,0,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.33 sec
 ARIMA(2,0,1)(0,0,0)[0] intercept   : AIC=23.765, Time=0.34 sec
 ARIMA(2,0,2)(0,0,0)[0] intercept   : AIC=inf, Time=0.44 sec
 ARIMA(1,0,2)(0,0,0)[0] intercept   : AIC=inf, Time=0.30 sec
 ARIMA(2,0,1)(0,0,0)[0]             : AIC=inf, Time=0.31 sec

Best model:  ARIMA(2,0,1)(0,0,0)[0] intercept
Total fit time: 2.178 seconds

Best SARIMA model: (2, 0, 1)
AIC: 23.765486321378994

Forecast samples:
2016 forecast: 6.79
2020 forecast: 6.93
2025 forecast: 6.93


C:\Users\shahe\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\shahe\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [12]:
# Evaluate on available actual values (2017-2023)
eval_df = df[(df["Region"] == "Central America") & df["Year"].between(2017, 2023)].copy()
eval_df["Undernourishment (%)"] = eval_df["Undernourishment (%)"].astype(float)

# Add SARIMA predictions
eval_df = eval_df.merge(
    central_america_sarima_forecast[["Year", "SARIMA"]],
    on="Year",
    how="left"
)

# Calculate simple baseline (last known value)
last_value = train["Undernourishment (%)"].iloc[-1]
eval_df["Baseline"] = last_value

# Calculate metrics
y_true = eval_df["Undernourishment (%)"]
y_pred = eval_df["SARIMA"]
y_baseline = eval_df["Baseline"]

# SARIMA metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)

# R-squared calculation with robust handling
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean)**2)
ss_res = np.sum((y_true - y_pred)**2)

try:
    r2 = 1 - (ss_res / ss_tot)
except ZeroDivisionError:
    r2 = np.nan

# Baseline metrics
baseline_mae = mean_absolute_error(y_true, y_baseline)
baseline_rmse = mean_squared_error(y_true, y_baseline, squared=False)

# Print metrics
print("Central America - SARIMA Evaluation Metrics")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Print baseline comparison
print(f"\nBaseline (constant) model metrics:")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")

# Create comparison table
comparison = pd.DataFrame({
    "Year": eval_df["Year"],
    "Actual": eval_df["Undernourishment (%)"],
    "SARIMA": eval_df["SARIMA"],
    "Error": eval_df["Undernourishment (%)"] - eval_df["SARIMA"]
})
print("\nYear-by-year comparison:")
print(comparison)

Central America - SARIMA Evaluation Metrics
MAE: 1.1040
RMSE: 1.1165
R²: -50.7603

Baseline (constant) model metrics:
MAE: 0.5857
RMSE: 0.6059

Year-by-year comparison:
   Year  Actual    SARIMA     Error
0  2017     6.0  6.876854 -0.876854
1  2018     6.0  6.912388 -0.912388
2  2019     5.6  6.923630 -1.323630
3  2020     5.6  6.927556 -1.327556
4  2021     5.8  6.928874 -1.128874
5  2022     5.9  6.929324 -1.029324
6  2023     5.8  6.929476 -1.129476


C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [13]:
# SARIMA model for South America
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Prepare training data (2001-2015)
train = df[(df["Region"] == "South America") & (df["Year"] >= 2001) & (df["Year"] <= 2015)].copy()
train["Undernourishment (%)"] = train["Undernourishment (%)"].astype(float)

# Check data characteristics
print("South America training data summary:")
print(f"Number of years: {len(train)}")
print(f"Min value: {train['Undernourishment (%)'].min():.2f}")
print(f"Max value: {train['Undernourishment (%)'].max():.2f}")
print(f"Mean value: {train['Undernourishment (%)'].mean():.2f}")

# Fit SARIMA model with improved configuration
model = auto_arima(
    train["Undernourishment (%)"],
    seasonal=True,
    m=1,                # Annual data
    start_p=0, max_p=2, # Limit complexity
    start_q=0, max_q=2,
    d=None,             # Auto-detect differencing
    max_d=1,            # Limit differencing
    trace=True,
    suppress_warnings=True,
    stepwise=True,
    error_action='ignore'
)

# Print model details
print(f"\nBest SARIMA model: {model.order}")
print(f"AIC: {model.aic()}")

# Forecast 10 years (2016-2025)
forecast = model.predict(n_periods=10)

# Ensure forecasts are not negative
forecast = np.maximum(forecast, train["Undernourishment (%)"].min())

# Store in DataFrame
south_america_sarima_forecast = pd.DataFrame({
    "Year": list(range(2016, 2026)),
    "SARIMA": forecast
})

# Print forecast samples
print("\nForecast samples:")
print(f"2016: {south_america_sarima_forecast['SARIMA'].iloc[0]:.2f}%")
print(f"2020: {south_america_sarima_forecast['SARIMA'].iloc[4]:.2f}%")
print(f"2025: {south_america_sarima_forecast['SARIMA'].iloc[9]:.2f}%")

South America training data summary:
Number of years: 15
Min value: 3.60
Max value: 11.10
Mean value: 6.80
Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=29.395, Time=0.02 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=24.764, Time=0.08 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=26.455, Time=0.10 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=34.139, Time=0.05 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=26.529, Time=0.12 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=26.640, Time=0.17 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=inf, Time=0.47 sec
 ARIMA(1,1,0)(0,0,0)[0]             : AIC=35.815, Time=0.04 sec

Best model:  ARIMA(1,1,0)(0,0,0)[0] intercept
Total fit time: 1.057 seconds

Best SARIMA model: (1, 1, 0)
AIC: 24.763578007428418

Forecast samples:
2016: 3.60%
2020: 3.60%
2025: 3.60%


C:\Users\shahe\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\shahe\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [14]:
# Prepare test data (2017-2023)
eval_df = df[(df["Region"] == "South America") & df["Year"].between(2017, 2023)].copy()
eval_df["Undernourishment (%)"] = eval_df["Undernourishment (%)"].astype(float)

# Add SARIMA predictions
eval_df = eval_df.merge(
    south_america_sarima_forecast[["Year", "SARIMA"]],
    on="Year",
    how="left"
)

# Add a simple baseline model (constant prediction)
baseline_value = train["Undernourishment (%)"].iloc[-1]  # Last observed value
eval_df["Baseline"] = baseline_value

# Calculate metrics
y_true = eval_df["Undernourishment (%)"]
y_pred = eval_df["SARIMA"]
y_baseline = eval_df["Baseline"]

# SARIMA metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)

# R-squared calculation with robust handling
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean)**2)
ss_res = np.sum((y_true - y_pred)**2)

try:
    r2 = 1 - (ss_res / ss_tot)
except ZeroDivisionError:
    r2 = np.nan

# Baseline metrics
baseline_mae = mean_absolute_error(y_true, y_baseline)
baseline_rmse = mean_squared_error(y_true, y_baseline, squared=False)

# Print SARIMA metrics
print("South America - SARIMA Evaluation Metrics")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Print baseline comparison
print(f"\nBaseline model metrics:")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")

# Calculate absolute errors
eval_df["Abs_Error"] = abs(eval_df["Undernourishment (%)"] - eval_df["SARIMA"])
eval_df["Baseline_Abs_Error"] = abs(eval_df["Undernourishment (%)"] - eval_df["Baseline"])

# Print comparison table
results = pd.DataFrame({
    "Year": eval_df["Year"],
    "Actual": eval_df["Undernourishment (%)"],
    "SARIMA": eval_df["SARIMA"],
    "Error": eval_df["Abs_Error"]
})
print("\nYear-by-year comparison:")
print(results)

# Check for significant improvements
years_improved = sum(eval_df["Abs_Error"] < eval_df["Baseline_Abs_Error"])
print(f"\nSARIMA outperformed baseline in {years_improved} out of {len(eval_df)} years")

South America - SARIMA Evaluation Metrics
MAE: 1.8571
RMSE: 1.9508
R²: -9.6682

Baseline model metrics:
MAE: 1.5571
RMSE: 1.6678

Year-by-year comparison:
   Year  Actual  SARIMA  Error
0  2017     4.9     3.6    1.3
1  2018     5.0     3.6    1.4
2  2019     4.8     3.6    1.2
3  2020     5.9     3.6    2.3
4  2021     6.5     3.6    2.9
5  2022     5.9     3.6    2.3
6  2023     5.2     3.6    1.6

SARIMA outperformed baseline in 0 out of 7 years


C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [139]:
!pip install prophet

In [15]:
# Prophet model for Central Asia
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Filter and format training data (2001-2015)
train = df[(df["Region"] == "Central Asia") & (df["Year"] >= 2001) & (df["Year"] <= 2015)].copy()
train["Undernourishment (%)"] = train["Undernourishment (%)"].astype(float)

# Check training data
print("Central Asia training data for Prophet:")
print(f"Years: {min(train['Year'])} to {max(train['Year'])}")
print(f"Undernourishment range: {train['Undernourishment (%)'].min():.2f}% to {train['Undernourishment (%)'].max():.2f}%")

# Prepare data for Prophet (requires ds and y columns)
train_prophet = train.rename(columns={"Year": "ds", "Undernourishment (%)": "y"})
train_prophet["ds"] = pd.to_datetime(train_prophet["ds"], format="%Y")

# Configure and fit Prophet model
model = Prophet(
    yearly_seasonality=False,  # No yearly seasonality for annual data
    daily_seasonality=False,   # No daily seasonality
    changepoint_prior_scale=0.1,  # Allow moderate flexibility in trend changes
    changepoint_range=0.9      # Allow changepoints throughout most of the range
)

# Add model components
model.add_seasonality(
    name='custom',
    period=5,          # Look for 5-year cycles
    fourier_order=1    # Simple cycle
)

# Fit the model
model.fit(train_prophet)

# Generate future dataframe for forecasting
future = model.make_future_dataframe(periods=10, freq='Y')

# Make predictions
forecast = model.predict(future)

# Extract and format forecast for 2016-2025
forecast["Year"] = forecast["ds"].dt.year
predicted = forecast[forecast["Year"].between(2016, 2025)][["Year", "yhat", "yhat_lower", "yhat_upper"]]

# Ensure no negative predictions (undernourishment can't be negative)
predicted["yhat"] = predicted["yhat"].clip(lower=0)
predicted["yhat_lower"] = predicted["yhat_lower"].clip(lower=0)

# Rename for consistency with other models
central_asia_prophet_forecast = predicted.rename(columns={
    "yhat": "Prophet",
    "yhat_lower": "Lower_CI", 
    "yhat_upper": "Upper_CI"
})

# Display forecast statistics
print("\nForecast statistics:")
print(f"Min forecast: {central_asia_prophet_forecast['Prophet'].min():.2f}%")
print(f"Max forecast: {central_asia_prophet_forecast['Prophet'].max():.2f}%")
print(f"Average forecast: {central_asia_prophet_forecast['Prophet'].mean():.2f}%")

Central Asia training data for Prophet:
Years: 2001 to 2015
Undernourishment range: 3.90% to 17.30%


10:36:21 - cmdstanpy - INFO - Chain [1] start processing
10:36:21 - cmdstanpy - INFO - Chain [1] done processing



Forecast statistics:
Min forecast: 0.00%
Max forecast: 1.68%
Average forecast: 0.30%


C:\Users\shahe\anaconda3\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(


In [17]:
# Prepare evaluation data (2017-2023)
eval_df = df[(df["Region"] == "Central Asia") & df["Year"].between(2017, 2023)].copy()
eval_df["Undernourishment (%)"] = eval_df["Undernourishment (%)"].astype(float)

# Add Prophet predictions
eval_df = eval_df.merge(
    central_asia_prophet_forecast[["Year", "Prophet"]],
    on="Year", 
    how="left"
)

# Create simple baseline model (last observed value)
last_value = train["Undernourishment (%)"].iloc[-1]
eval_df["Baseline"] = last_value

# Calculate metrics
y_true = eval_df["Undernourishment (%)"]
y_pred = eval_df["Prophet"]
y_baseline = eval_df["Baseline"]

# Prophet metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)

# R-squared calculation with robust handling
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean)**2)
ss_res = np.sum((y_true - y_pred)**2)

try:
    r2 = 1 - (ss_res / ss_tot)
except ZeroDivisionError:
    r2 = np.nan

# Baseline metrics
baseline_mae = mean_absolute_error(y_true, y_baseline)
baseline_rmse = mean_squared_error(y_true, y_baseline, squared=False)

# Print Prophet metrics
print("Central Asia - Prophet Evaluation Metrics")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Print baseline comparison
print(f"\nBaseline model metrics:")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")

# Create comparison table
results = pd.DataFrame({
    "Year": eval_df["Year"],
    "Actual": eval_df["Undernourishment (%)"],
    "Prophet": eval_df["Prophet"],
    "Error": eval_df["Undernourishment (%)"] - eval_df["Prophet"]
})
print("\nYear-by-year comparison:")
print(results)

Central Asia - Prophet Evaluation Metrics
MAE: 2.9058
RMSE: 2.9138
R²: -148.6458

Baseline model metrics:
MAE: 0.8429
RMSE: 0.8759

Year-by-year comparison:
   Year  Actual   Prophet     Error
0  2017     3.4  0.807489  2.592511
1  2018     2.9  0.000000  2.900000
2  2019     2.6  0.000000  2.600000
3  2020     3.2  0.252180  2.947820
4  2021     3.2  0.000000  3.200000
5  2022     3.1  0.000000  3.100000
6  2023     3.0  0.000000  3.000000


C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [18]:
# Prophet model for Central America
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Prepare training data (2001-2015)
train = df[(df["Region"] == "Central America") & (df["Year"].between(2001, 2015))].copy()
train["Undernourishment (%)"] = train["Undernourishment (%)"].astype(float)

# Examine training data
print("Central America training data:")
print(f"Number of years: {len(train)}")
print(f"Year range: {train['Year'].min()} to {train['Year'].max()}")
print(f"Undernourishment range: {train['Undernourishment (%)'].min():.2f}% to {train['Undernourishment (%)'].max():.2f}%")
print(f"Trend direction: {'Decreasing' if train['Undernourishment (%)'].iloc[0] > train['Undernourishment (%)'].iloc[-1] else 'Increasing'}")

# Format data for Prophet (requires ds and y columns)
train_prophet = train.rename(columns={"Year": "ds", "Undernourishment (%)": "y"})
train_prophet["ds"] = pd.to_datetime(train_prophet["ds"], format="%Y")

# Configure Prophet model with parameters suited to Central America's data
model = Prophet(
    yearly_seasonality=False,        # No yearly seasonality for annual data
    daily_seasonality=False,         # No daily seasonality
    changepoint_prior_scale=0.1      # Allow some flexibility for trend changes
)

# Fit the model
model.fit(train_prophet)

# Create future dates for forecasting
future = model.make_future_dataframe(periods=10, freq='Y')

# Generate forecast
forecast = model.predict(future)

# Extract forecast for 2016-2025
forecast["Year"] = forecast["ds"].dt.year
forecast_subset = forecast[forecast["Year"].between(2016, 2025)]

# Extract relevant columns
predicted = forecast_subset[["Year", "yhat"]]

# Ensure no negative values (undernourishment can't be negative)
predicted["yhat"] = predicted["yhat"].clip(lower=0)

# Rename columns for consistency
central_america_prophet_forecast = predicted.rename(columns={"yhat": "Prophet"})

# Display forecast summary
print("\nCentral America Prophet forecast summary:")
print(f"2016 forecast: {central_america_prophet_forecast['Prophet'].values[0]:.2f}%")
print(f"2020 forecast: {central_america_prophet_forecast['Prophet'].values[4]:.2f}%")
print(f"2025 forecast: {central_america_prophet_forecast['Prophet'].values[-1]:.2f}%")

10:38:40 - cmdstanpy - INFO - Chain [1] start processing
10:38:41 - cmdstanpy - INFO - Chain [1] done processing


Central America training data:
Number of years: 15
Year range: 2001 to 2015
Undernourishment range: 6.20% to 8.20%
Trend direction: Decreasing

Central America Prophet forecast summary:
2016 forecast: 6.19%
2020 forecast: 5.86%
2025 forecast: 5.52%


C:\Users\shahe\anaconda3\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(
C:\Users\shahe\AppData\Local\Temp\ipykernel_12124\750257675.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  predicted["yhat"] = predicted["yhat"].clip(lower=0)


In [19]:
# Prepare evaluation data (2017-2023 actuals)
eval_df = df[(df["Region"] == "Central America") & df["Year"].between(2017, 2023)].copy()
eval_df["Undernourishment (%)"] = eval_df["Undernourishment (%)"].astype(float)

# Add Prophet forecasts
eval_df = eval_df.merge(
    central_america_prophet_forecast[["Year", "Prophet"]],
    on="Year",
    how="left"
)

# Add simple trend baseline (last value)
last_value = train["Undernourishment (%)"].iloc[-1]
eval_df["Baseline"] = last_value

# Calculate metrics
y_true = eval_df["Undernourishment (%)"]
y_pred = eval_df["Prophet"]
y_baseline = eval_df["Baseline"]

# Prophet metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)

# R-squared calculation with robust handling
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean)**2)
ss_res = np.sum((y_true - y_pred)**2)

try:
    r2 = 1 - (ss_res / ss_tot)
except ZeroDivisionError:
    r2 = np.nan

# Baseline metrics
baseline_mae = mean_absolute_error(y_true, y_baseline)
baseline_rmse = mean_squared_error(y_true, y_baseline, squared=False)

# Print Prophet metrics
print("Central America - Prophet Evaluation Metrics")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Print baseline comparison
print(f"\nBaseline model metrics:")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")

# Calculate prediction errors
eval_df["Abs_Error"] = abs(eval_df["Undernourishment (%)"] - eval_df["Prophet"])
eval_df["Pct_Error"] = 100 * eval_df["Abs_Error"] / eval_df["Undernourishment (%)"]

# Create comparison table
results = pd.DataFrame({
    "Year": eval_df["Year"],
    "Actual": eval_df["Undernourishment (%)"],
    "Prophet": eval_df["Prophet"],
    "Error": eval_df["Abs_Error"],
    "Error_%": eval_df["Pct_Error"].round(1)
})
print("\nYear-by-year comparison:")
print(results[["Year", "Actual", "Prophet", "Error"]])

# Identify years with largest errors
max_error_year = eval_df.loc[eval_df["Abs_Error"].idxmax(), "Year"]
print(f"\nLargest error in year: {max_error_year}")

Central America - Prophet Evaluation Metrics
MAE: 0.1653
RMSE: 0.1984
R²: -0.6349

Baseline model metrics:
MAE: 0.5857
RMSE: 0.6059

Year-by-year comparison:
   Year  Actual   Prophet     Error
0  2017     6.0  6.106334  0.106334
1  2018     6.0  6.022928  0.022928
2  2019     5.6  5.939521  0.339521
3  2020     5.6  5.855886  0.255886
4  2021     5.8  5.772480  0.027520
5  2022     5.9  5.689074  0.210926
6  2023     5.8  5.605667  0.194333

Largest error in year: 2019


C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [20]:
# Prophet model for South America
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Prepare training data (2001-2015)
train = df[(df["Region"] == "South America") & (df["Year"].between(2001, 2015))].copy()
train["Undernourishment (%)"] = train["Undernourishment (%)"].astype(float)

# Examine training data
print("South America training data:")
print(f"Number of years: {len(train)}")
print(f"Year range: {train['Year'].min()} to {train['Year'].max()}")
print(f"Undernourishment range: {train['Undernourishment (%)'].min():.2f}% to {train['Undernourishment (%)'].max():.2f}%")
print(f"Average: {train['Undernourishment (%)'].mean():.2f}%")

# Format data for Prophet (requires ds and y columns)
train_prophet = train.rename(columns={"Year": "ds", "Undernourishment (%)": "y"})
train_prophet["ds"] = pd.to_datetime(train_prophet["ds"], format="%Y")

# Configure Prophet model 
model = Prophet(
    yearly_seasonality=False,      # No yearly seasonality for annual data
    daily_seasonality=False,       # No daily seasonality
    changepoint_prior_scale=0.05,  # Lower flexibility for stable series
    n_changepoints=5              # Limit number of changepoints for smoother forecast
)

# Fit the model
model.fit(train_prophet)

# Create future dates for forecasting
future = model.make_future_dataframe(periods=10, freq='Y')

# Generate forecast
forecast = model.predict(future)

# Extract forecast for 2016-2025
forecast["Year"] = forecast["ds"].dt.year
predicted = forecast[forecast["Year"].between(2016, 2025)][["Year", "yhat"]]

# Create a clean copy to avoid warnings
predicted = predicted.copy()

# Ensure no negative values and set reasonable minimum based on history
min_value = train["Undernourishment (%)"].min()
predicted["yhat"] = predicted["yhat"].clip(lower=min_value)

# Rename for consistency
south_america_prophet_forecast = predicted.rename(columns={"yhat": "Prophet"})

# Display forecast summary
print("\nSouth America Prophet forecast summary:")
print(f"2016 forecast: {south_america_prophet_forecast['Prophet'].values[0]:.2f}%")
print(f"2020 forecast: {south_america_prophet_forecast['Prophet'].values[4]:.2f}%")
print(f"2025 forecast: {south_america_prophet_forecast['Prophet'].values[-1]:.2f}%")

10:41:37 - cmdstanpy - INFO - Chain [1] start processing
10:41:37 - cmdstanpy - INFO - Chain [1] done processing


South America training data:
Number of years: 15
Year range: 2001 to 2015
Undernourishment range: 3.60% to 11.10%
Average: 6.80%

South America Prophet forecast summary:
2016 forecast: 3.60%
2020 forecast: 3.60%
2025 forecast: 3.60%


C:\Users\shahe\anaconda3\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(


In [21]:
# Prepare evaluation data (2017-2023)
eval_df = df[(df["Region"] == "South America") & df["Year"].between(2017, 2023)].copy()
eval_df["Undernourishment (%)"] = eval_df["Undernourishment (%)"].astype(float)

# Add Prophet predictions
eval_df = eval_df.merge(
    south_america_prophet_forecast[["Year", "Prophet"]],
    on="Year",
    how="left"
)

# Add naive baseline prediction (last observed value)
last_value = train["Undernourishment (%)"].iloc[-1]
eval_df["Baseline"] = last_value

# Calculate metrics
y_true = eval_df["Undernourishment (%)"]
y_pred = eval_df["Prophet"]
y_baseline = eval_df["Baseline"]

# Prophet metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred, squared=False)

# R-squared calculation with robust handling
y_mean = np.mean(y_true)
ss_tot = np.sum((y_true - y_mean)**2)
ss_res = np.sum((y_true - y_pred)**2)

try:
    r2 = 1 - (ss_res / ss_tot)
except ZeroDivisionError:
    r2 = np.nan

# Baseline metrics
baseline_mae = mean_absolute_error(y_true, y_baseline)
baseline_rmse = mean_squared_error(y_true, y_baseline, squared=False)

# Print Prophet metrics
print("South America - Prophet Evaluation Metrics")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Print baseline comparison
print(f"\nBaseline model (last value) metrics:")
print(f"MAE: {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")

# Create comparison table
results = pd.DataFrame({
    "Year": eval_df["Year"],
    "Actual": eval_df["Undernourishment (%)"],
    "Prophet": eval_df["Prophet"],
    "Error": eval_df["Undernourishment (%)"] - eval_df["Prophet"]
})
print("\nYear-by-year comparison:")
print(results[["Year", "Actual", "Prophet", "Error"]])

South America - Prophet Evaluation Metrics
MAE: 1.8571
RMSE: 1.9508
R²: -9.6682

Baseline model (last value) metrics:
MAE: 1.5571
RMSE: 1.6678

Year-by-year comparison:
   Year  Actual  Prophet  Error
0  2017     4.9      3.6    1.3
1  2018     5.0      3.6    1.4
2  2019     4.8      3.6    1.2
3  2020     5.9      3.6    2.3
4  2021     6.5      3.6    2.9
5  2022     5.9      3.6    2.3
6  2023     5.2      3.6    1.6


C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\shahe\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [22]:
import pandas as pd
# Load partial file
df = pd.read_csv("all_model_predictions_partial.csv")

# Filter and split per region
central_asia = df[df["Country"] == "Central Asia"].copy()
central_america = df[df["Country"] == "Central America"].copy()
south_america = df[df["Country"] == "South America"].copy()

# Merge SARIMA forecasts
central_asia = pd.merge(central_asia, central_asia_sarima_forecast, on="Year", how="left")
central_america = pd.merge(central_america, central_america_sarima_forecast, on="Year", how="left")
south_america = pd.merge(south_america, south_america_sarima_forecast, on="Year", how="left")

# Merge Prophet forecasts
central_asia = pd.merge(central_asia, central_asia_prophet_forecast, on="Year", how="left")
central_america = pd.merge(central_america, central_america_prophet_forecast, on="Year", how="left")
south_america = pd.merge(south_america, south_america_prophet_forecast, on="Year", how="left")

# Combine everything
final_df = pd.concat([central_asia, central_america, south_america], ignore_index=True)

# Clean up column names to ensure consistency
for col in final_df.columns:
    if 'sarima' in col.lower():
        final_df.rename(columns={col: "SARIMA"}, inplace=True)
    elif 'prophet' in col.lower():
        final_df.rename(columns={col: "Prophet"}, inplace=True)

# Save final CSV
final_df.to_csv("all_model_predictions_final.csv", index=False)